[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/google-deepmind/regress-lm/blob/main/docs/train_density_example.ipynb)

# Multi-objective density learning with Regression Language Models

Demonstrates that a small `RegressLM` can learn and capture arbitrary multi-objective densities $p(y_1, y_2 \mid x)$ by training on infinite synthetic samples from parameterised 2D shapes.

This notebook is a minimal, quick-start demonstration designed to run in about 20 minutes on a Colab T4 GPU. Models trained for longer periods or with larger capacities will produce significantly sharper and more detailed geometric boundaries.


In [ ]:
!pip install -q git+https://github.com/google-deepmind/regress-lm

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
from regress_lm import core
from regress_lm.rlm import RegressLM
import torch
from torch.utils.data import DataLoader, IterableDataset

FIELD_SEP = " "
NUM_DECIMALS = 2

## Shape Samplers

Let's define our sampling functions for 4 basic shapes: `spiral`, `hollow_square`, `zigzag`, and `half_moons`.

Our RegressLM's task is essentially a text-conditional regression problem. Based on a prompt configuring the shape (name, coordinate centers $x$ and $y$, and eccentricity), it must predict $(x, y)$ coordinate samples that follow the given density.

In [ ]:
SHAPES = ["spiral", "hollow_square", "zigzag", "half_moons"]


def sample_spiral(
    rng: np.random.Generator, n: int = 1, scale: float = 0.3
) -> tuple[np.ndarray, np.ndarray]:
  """Archimedean spiral:  r = scale * theta / (4*pi)."""
  theta = rng.uniform(0, 4 * np.pi, size=n)
  r = scale * theta / (4 * np.pi)
  return r * np.cos(theta), r * np.sin(theta)


def sample_hollow_square(
    rng: np.random.Generator, n: int = 1, half_side: float = 0.3
) -> tuple[np.ndarray, np.ndarray]:
  """Uniform points on the four edges of a square."""
  edge = rng.integers(0, 4, size=n)
  t = rng.uniform(-half_side, half_side, size=n)
  dx = np.where(
      edge == 0,
      t,
      np.where(edge == 1, t, np.where(edge == 2, -half_side, half_side)),
  )
  dy = np.where(
      edge == 0,
      half_side,
      np.where(edge == 1, -half_side, np.where(edge == 2, t, t)),
  )
  return dx, dy


def sample_zigzag(
    rng: np.random.Generator,
    n: int = 1,
    x_range: float = 0.3,
    period: float = 0.15,
    amplitude: float = 0.15,
) -> tuple[np.ndarray, np.ndarray]:
  """Triangle-wave zigzag:  y = amplitude * triangle(x / period)."""
  dx = rng.uniform(-x_range, x_range, size=n)
  phase = dx / period - np.floor(dx / period + 0.5)
  dy = amplitude * (2 * np.abs(2 * phase) - 1)
  return dx, dy


def sample_half_moons(
    rng: np.random.Generator, n: int = 1, radius: float = 0.3
) -> tuple[np.ndarray, np.ndarray]:
  """Two interleaving half-moons (sklearn make_moons style)."""
  is_upper = rng.integers(0, 2, size=n).astype(bool)
  angle = rng.uniform(0, np.pi, size=n)
  dx = np.where(
      is_upper, radius * np.cos(angle), radius * np.cos(angle) + radius * 0.5
  )
  dy = np.where(
      is_upper, radius * np.sin(angle), -radius * np.sin(angle) + radius * 0.1
  )
  return dx, dy


SHAPE_SAMPLERS = {
    "spiral": sample_spiral,
    "hollow_square": sample_hollow_square,
    "zigzag": sample_zigzag,
    "half_moons": sample_half_moons,
}


def sample_from_shape(
    shape: str,
    cx: float,
    cy: float,
    eccentricity: float,
    rng: np.random.Generator,
    n: int = 1,
) -> tuple[np.ndarray, np.ndarray]:
  """Sample n points from the parameterised density."""
  dx, dy = SHAPE_SAMPLERS[shape](rng, n=n)

  dx = dx / eccentricity
  noise = 0.05
  sigma = noise / 1.96
  dx = dx + rng.normal(0, sigma, size=n)
  dy = dy + rng.normal(0, sigma, size=n)
  return cx + dx, cy + dy


def make_input_string(
    shape: str, cx: float, cy: float, eccentricity: float
) -> str:
  w = 3 + NUM_DECIMALS  # e.g. 5 for NUM_DECIMALS=2: "+5.00"
  fmt = f"+0{w}.{NUM_DECIMALS}f"
  return (
      f"{shape} X:{FIELD_SEP}{cx:{fmt}} Y:{FIELD_SEP}{cy:{fmt}}"
      f" E:{FIELD_SEP}{int(eccentricity)}"
  )


def parse_input_string(s: str) -> tuple[str, float, float, float]:
  """Parse a labeled input string back into (shape, cx, cy, ecc)."""
  parts = s.split()
  return (
      parts[0],
      float(parts[2]),
      float(parts[4]),
      float(parts[6]),
  )

## Infinite Streaming Dataset

We construct an `IterableDataset` that generates examples on the fly. We sample the config uniformly within random ranges, construct the prompt textual string, and pair it with a single random sample coordinate.

In [ ]:
class DensityDataset(IterableDataset):
  """Yields ``core.Example(x=input_str, y=[y1, y2])`` forever.

  Each example randomly draws shape / centre / eccentricity,
  then samples a chunk of (y1, y2) points from that distribution.
  """

  def __init__(
      self,
      seed: int = 42,
      cx_range: tuple[float, float] = (-5.0, 5.0),
      cy_range: tuple[float, float] = (-3.0, 3.0),
      ecc_choices: list[int] = [1, 2, 3],
      chunk_size: int = 16,
  ):
    self.seed = seed
    self.cx_range = cx_range
    self.cy_range = cy_range
    self.ecc_choices = ecc_choices
    self.chunk_size = chunk_size

  def __iter__(self):
    worker_info = torch.utils.data.get_worker_info()
    worker_seed = self.seed + (worker_info.id if worker_info else 0)
    rng = np.random.default_rng(worker_seed)

    while True:
      shape = rng.choice(SHAPES)
      cx = round(float(rng.uniform(*self.cx_range)), NUM_DECIMALS)
      cy = round(float(rng.uniform(*self.cy_range)), NUM_DECIMALS)
      ecc = float(rng.choice(self.ecc_choices))

      x_str = make_input_string(shape, cx, cy, ecc)
      y1, y2 = sample_from_shape(shape, cx, cy, ecc, rng, n=self.chunk_size)

      for i in range(self.chunk_size):
        yield core.Example(x=x_str, y=[float(y1[i]), float(y2[i])])

## Evaluation Helpers

During evaluation, we test how well `RegressLM` has learned to model coordinates conditionally by checking multiple shape arrangements. Let's list those test prompts out and visualise the ground truth samples.

In [ ]:
_EVAL_CONFIGS = [
    # (shape, cx, cy, eccentricity)
    # ecc=1 (near centre)
    ("spiral", -1.0, 0.5, 1),
    ("hollow_square", 1.0, 0.5, 1),
    ("zigzag", -1.0, -0.5, 1),
    ("half_moons", 1.0, -0.5, 1),
    # ecc=3 (further out)
    ("spiral", -2.0, 0.5, 3),
    ("hollow_square", 2.0, 0.5, 3),
    ("zigzag", -2.0, -0.5, 3),
    ("half_moons", 2.0, -0.5, 3),
]
EVAL_PROMPTS = [make_input_string(*cfg) for cfg in _EVAL_CONFIGS]

EVAL_XLIM = (-2.5, 2.5)
EVAL_YLIM = (-1.25, 1.25)

SHAPE_COLORS = {
    "spiral": "tab:blue",
    "hollow_square": "tab:orange",
    "zigzag": "tab:green",
    "half_moons": "tab:red",
}


def plot_samples(
    results: dict[str, np.ndarray], step: int | str = "Ground Truth"
):
  fig, ax = plt.subplots(figsize=(8, 8))
  handles = {}

  for prompt, pts in results.items():
    shape, cx, cy, _ = parse_input_string(prompt)
    color = SHAPE_COLORS.get(shape, "tab:gray")
    h = ax.scatter(
        pts[:, 0],
        pts[:, 1],
        s=1.5,
        alpha=0.45,
        linewidths=0,
        color=color,
        rasterized=True,
    )
    ax.plot(cx, cy, "x", color="k", ms=7, mew=1.2)
    if shape not in handles:
      handles[shape] = h

  ax.legend(
      handles.values(),
      handles.keys(),
      fontsize=11,
      markerscale=6,
      framealpha=0.85,
      loc="upper center",
  )
  ax.set_xlim(EVAL_XLIM)
  ax.set_ylim(EVAL_YLIM)
  ax.set_aspect("equal")
  ax.grid(True, alpha=0.2, lw=0.3)
  ax.set_title(
      f"{step}\n(× = centre;  inner: ecc=1,  outer: ecc=3)",
      fontsize=13,
      weight="bold",
  )
  plt.show()


# Visualize Ground Truth
rng = np.random.default_rng(0)
ground_truth_res = {}
for prompt in EVAL_PROMPTS:
  shape, cx, cy, ecc = parse_input_string(prompt)
  y1, y2 = sample_from_shape(shape, cx, cy, ecc, rng, n=3000)
  ground_truth_res[prompt] = np.stack([y1, y2], axis=1)

plot_samples(ground_truth_res, step="Ground Truth Densities")

## Model Initialization

We're going to use the `RegressLM` class to create our model from scratch.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Model config
d_model = 256
num_encoder_layers = 4
num_decoder_layers = 4

rlm = RegressLM.from_scratch(
    device=device,
    d_model=d_model,
    num_encoder_layers=num_encoder_layers,
    num_decoder_layers=num_decoder_layers,
    max_num_objs=2,  # multi-objective: (y1, y2)
    max_input_len=48,  # labeled format: "spiral X: -1.00 Y: +0.50 E: 1"
    compile_model=False,
)
model = rlm.model
model.to(device)

num_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters : {num_params:,}")

## Training Loop

We train by optimising the predicted continuous samples given the textual prompt via our AdamW optimiser and cosine annealing learning rate scheduler.
*(For demonstration in this notebook, we'll run 2,000 steps which takes roughly 20 minutes on a T4 GPU)*

In [ ]:
from regress_lm.pytorch import optimizers as rlm_optimizers
from tqdm.auto import tqdm

seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

batch_size = 2048
total_steps = 2000
warmup_steps = 200
log_every = 100
val_every = 500

lr = 2e-3
weight_decay = 0.0

# Training Dataset
ds = DensityDataset(seed=seed)
dl = DataLoader(
    ds,
    batch_size=batch_size,
    num_workers=2,
    collate_fn=model.converter.convert_examples,
    pin_memory=(device == "cuda"),
)
data_iter = iter(dl)

# Validation Dataset
val_ds = DensityDataset(seed=seed + 1)
val_dl = DataLoader(
    val_ds,
    batch_size=batch_size,
    num_workers=2,
    collate_fn=model.converter.convert_examples,
    pin_memory=(device == "cuda"),
)
val_iter = iter(val_dl)

scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda"))


def lr_lambda(step: int) -> float:
  if step <= warmup_steps:
    return 1.0
  return (warmup_steps / step) ** 0.5


optimizer = rlm_optimizers.muon_adamw(
    list(model.named_parameters()), lr=lr, weight_decay=weight_decay
)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

print("Starting training…")
model.train()
running_loss = 0.0

pbar = tqdm(total=total_steps)

for step in range(1, total_steps + 1):
  batch = next(data_iter)
  optimizer.zero_grad(set_to_none=True)

  with torch.autocast(
      device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")
  ):
    losses, metrics = model.compute_losses_and_metrics(batch)
    loss = losses.mean()

  scaler.scale(loss).backward()
  scaler.step(optimizer)
  scaler.update()
  scheduler.step()

  running_loss += metrics["loss_mean"].detach()

  if step % log_every == 0:
    avg_loss = (running_loss / log_every).item()
    lr_now = scheduler.get_last_lr()[0]
    ppl = math.exp(min(avg_loss, 20.0))

    pbar.update(log_every)
    pbar.set_postfix(
        {"loss": f"{avg_loss:.4f}", "ppl": f"{ppl:.2f}", "lr": f"{lr_now:.2e}"}
    )
    running_loss = 0.0

  if step % val_every == 0:
    model.eval()
    with torch.no_grad(), torch.autocast(
        device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")
    ):
      val_batch = next(val_iter)
      val_losses, _ = model.compute_losses_and_metrics(val_batch)
      val_loss = val_losses.mean().item()
      val_ppl = math.exp(min(val_loss, 20.0))
    model.train()

    tqdm.write(
        f"Step {step} | Val Loss: {val_loss:.4f} | Val PPL: {val_ppl:.2f}"
    )

pbar.close()

## Sampling

We evaluate the trained model on all eval prompts and compare against the ground truth plot from earlier.

In [ ]:
model.eval()
model_results = {}

print("Sampling from trained model...")

with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
  for prompt in EVAL_PROMPTS:
    xs = [core.ExampleInput(x=prompt)]
    samples = rlm.sample(xs, num_samples=3000)
    model_results[prompt] = np.array(samples[0])

plot_samples(model_results, step="Trained Model Densities")


Try generating samples for an arbitrary configuration prompt directly from the model!

In [ ]:
prompt = "hollow_square X: -1.00 Y: -0.50 E: 1"
xs = [core.ExampleInput(x=prompt)]

with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
  samples = rlm.sample(xs, num_samples=3000)

print("\nFirst 10 Generated (y1, y2) points:")
for pt in samples[0][:10]:
  print(f"  {pt[0]:.4f}, {pt[1]:.4f}")

results = {prompt: np.array(samples[0])}
plot_samples(results, step="Model Samples")